# ML-03 — Frame Your Lane as an ML Task



## 1. My Lane as an ML Task

Lane: Refresh / Content Opportunity Scoring

Task type: **Scoring / Ranking** (with a classification model underneath).

The core ML task is binary classification — predicting whether a page is
"decline-risk" or not — but the actual deliverable isn't the raw prediction,
it's a **ranked queue**: pages ordered by score so a reviewer with limited
time works through the most promising candidates first. This is why it's
scoring/ranking, not pure classification: the output only matters in relative
order, not as an isolated yes/no per page.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or Proxy

Starter proxy target: `trend_direction == "down"` — a snapshot-based bucket
calculated from the current 90-day window, not a future outcome.

This is a **beginner proxy**, not my ideal capstone target. A stronger target
would be forward-looking:

**prior 90 days of features → decline (or recovery) over the next 30 days**

I'm starting with the proxy label this week since it's what the starter data
supports directly, but I'll move toward a true future-window label once I
have the warehouse release, so I'm not just predicting the present.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success Metric

Primary metric: **Precision@50** (or Precision@K generally).

Why: this isn't a generic accuracy problem — a reviewer only has capacity to
look at a limited number of pages (say, top 50). What matters is: of the top
50 pages the model flags, how many are genuinely worth reviewing? That's
exactly what Precision@K measures, and it directly matches how the output
gets used.

The starter pipeline already reports this: baseline rules scored 0.240
Precision@50, while the random forest scored 0.740 — meaning of the top 50
picks, the baseline got ~12 right, and the model got ~37 right.

Secondary: Average Precision (to judge the full ranking, not just the top K).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")

# One row = one content page (content_id), evaluated over a 90-day window
print("Unit of analysis: one row = one content page (content_id)")
print("Shape:", df.shape)
df[['content_id', 'client_id', 'impressions_90d', 'sessions_90d',
    'content_age_days', 'trend_direction']].head(10)

Unit of analysis: one row = one content page (content_id)
Shape: (30000, 44)


,content_id,client_id,impressions_90d,sessions_90d,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,141,down
3,content_331d6c4de07b,client_19581e27de,11751,78,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,263,down
5,content_d4084a4bc775,client_f369cb89fc,3970,5,147,down
6,content_9a34b442b552,client_8722616204,20,1,90,down
7,content_a63219c6e95a,client_19581e27de,1724,28,445,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,68,90,down
9,content_c27558df2b0c,client_19581e27de,1240,3,257,down


## 5. Why ML Beats a Fixed Rule Here

The starter pipeline directly tested this: a hand-written baseline rule
(combining visibility, freshness risk, position opportunity, and depth gap
with fixed weights) scored 0.240 Precision@50. A random forest trained on
the same signals scored 0.740 Precision@50 — roughly 3x better at picking
genuinely worthwhile pages for the top of the queue.

Why a fixed rule falls short: a hand-tuned formula assumes the same weights
apply to every page, but real pages combine signals (visibility, staleness,
position, engagement) in ways that interact non-linearly — e.g., a stale
page might only matter if it's also high-traffic, and a fixed weighted sum
can't capture that kind of conditional interaction the way a tree-based
model can. That's exactly the kind of pattern ML is suited to find and a
fixed rule isn't.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 6. Self-Check

**Did I name the task type?** Yes — scoring/ranking, built on top of a binary
classification model (decline-risk vs not).

**Did I name a target/proxy?** Yes — starting with the starter's proxy
(`trend_direction == "down"`), with a clear path to a stronger future-window
label (prior 90 days → next 30 days decline).

**Did I name a success metric?** Yes — Precision@50 as primary, tied directly
to how a reviewer with limited capacity actually uses the output.

**Did I show the unit of analysis as a real dataframe?** Yes — one row = one
content page (content_id), shown via the starter dataset.

**Did I explain why ML beats a fixed rule?** Yes — the starter pipeline's own
numbers show a learned model beating a hand-written rule by roughly 3x on
Precision@50, because real signals interact in ways a fixed weighted formula
can't capture.